<a href="https://colab.research.google.com/github/mayura-andrew/ml-experiments/blob/main/nlp_task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

df = pd.read_csv('Tweets.csv')

df = df[["airline_sentiment", "text"]]

df.head()

,airline_sentiment,text
0,neutral,@VirginAmerica What @dhepburn said.
1,positive,@VirginAmerica plus you've added commercials t...
2,neutral,@VirginAmerica I didn't today... Must mean I n...
3,negative,@VirginAmerica it's really aggressive to blast...
4,negative,@VirginAmerica and it's a really big bad thing...


In [4]:
import nltk
import string
import re
from nltk.stem.porter import PorterStemmer
from nltk.corpus import stopwords

# Download required NLTK data
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

ps = PorterStemmer()

def clean_text(text):
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http.?://[^]+[]?', '', text)

    # Tokenize
    text = nltk.word_tokenize(text)

    y = []
    # Remove stop words (and punctuation for cleaner tokens)
    for i in text:
        if i not in stopwords.words('english') and i not in string.punctuation:
            y.append(i)

    text = y[:]
    y.clear()

    # Stemming
    for i in text:
        y.append(ps.stem(i))

    return " ".join(y)

# Apply the clean_text function to the text column
# (Note: This step might take a few minutes depending on the dataset size)
print("Cleaning text, please wait...")
df['text_cleaned'] = df['text'].apply(clean_text)
print("Text cleaning complete!")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Cleaning text, please wait...
Text cleaning complete!


In [5]:
# step 3: feature extraction


from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Create TfidfVectorizer with max_features = 3000
tfidf = TfidfVectorizer(max_features=3000)

# Fit and transform the cleaned text, then convert to an array
X = tfidf.fit_transform(df['text_cleaned']).toarray()

# Convert the sentiment labels to an array
Y = df['airline_sentiment'].values

print("Shape of X:", X.shape)
print("Shape of Y:", Y.shape)

Shape of X: (14640, 3000)
Shape of Y: (14640,)


In [6]:
#train model
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=2)

# 1. train and evaluate multinomial naive bayes
#-------------------------
mnb = MultinomialNB()
mnb.fit(X_train, y_train)
y_pred_mnb = mnb.predict(X_test)
mnb_accuracy = accuracy_score(y_test, y_pred_mnb)


print(f"Multinomial Naive Bayes Accuracy: {mnb_accuracy * 100:.2f}%")


# ---------------------------------------------------------
# 2. Train and evaluate Random Forest Classifier
# ---------------------------------------------------------
model = RandomForestClassifier()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
rf_accuracy = accuracy_score(y_test, y_pred)

print(f"Random Forest Accuracy: {rf_accuracy * 100:.2f}%")

Multinomial Naive Bayes Accuracy: 72.30%
Random Forest Accuracy: 74.97%
